In [20]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import numpy as np
import scipy
import copy

import pyMKL
from pymklpardiso import PardisoSolver

from scipy.sparse import coo_matrix, block_diag, identity, hstack, csr_matrix, csc_matrix, vstack
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import time 
from time import perf_counter
import matplotlib as mpl

from pyiga import assemble, bspline, vform, geometry, vis, solvers, utils, topology, ieti, algebra, operators, adaptive
from pyiga import algebra_cy, ieti_cy, bspline_cy

from scipy.sparse.linalg import aslinearoperator as LinOp

np.set_printoptions(linewidth=100000)
np.set_printoptions(precision=5)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [115]:
kvs=2*(bspline.make_knots(5,0,1,11),)
geo=geometry.unit_square()
corners = assemble.boundary_dofs(kvs,m=0)

In [118]:
Kh = assemble.assemble('a*inner(grad(u),grad(v)) *dx', a=1e7, kvs=kvs, geo=geo)

In [119]:
B = np.zeros((8,Kh.shape[0]))
for i in range(4):
    B[i,corners[i]]=1
    B[4+i,assemble.boundary_dofs(kvs,bdspec=i,ravel=True)] = assemble.assemble('v * ds', kvs=kvs, geo=geo, boundary=i).ravel()
#B = B.reshape(1, -1) 
Ah = scipy.sparse.bmat([[Kh, B.T],
                        [B, None]], format="csr")
# Ah=Kh

In [120]:
x = np.random.rand(Ah.shape[1])
b = Ah@x

t = time.time()
solver=scipy.sparse.linalg.splu(Ah.tocsc())
print(np.linalg.norm(Ah@solver.solve(b)-b))
print('time: '+str(time.time()-t))

t = time.time()
solver = operators.make_solver(Ah,symm=True,spd=False)
print(np.linalg.norm(Ah@solver.solve(b)-b))
print('time: '+str(time.time()-t))

# t=time.time()
# solver = pyMKL.pardisoSolver(Ah, mtype=11)
# solver.factor()
# print(np.linalg.norm(Ah@solver.solve(b)-b))
# print('time: '+str(time.time()-t))

3.233344261636344e-08
time: 0.009952783584594727
4.183098725621274e-08
time: 0.0029532909393310547


In [83]:
a,b = False,False

In [84]:
if a:
    print(1)
elif b:
    print(2)
else:
    print(3)

3


In [ ]:
solver = PardisoSolver(A, mtype=11)
print()

In [19]:
np.linalg.eigh(Ah.toarray())

(array([-0.78789,  0.00794,  0.1364 ,  0.15999,  0.23197,  0.31877,  0.37001,  0.37812,  0.39243,  0.4378 ,  0.4418 ,  0.45224,  0.47686,  0.4961 ,  0.52424,  0.53278,  0.53696,  0.5547 ,  0.56014,  0.59289,  0.61153,  0.67153,  0.68501,  0.7185 ,  0.71956,  0.77882,  0.8588 ,  0.86557,  0.91493,  0.99363,  1.00579,  1.01683,  1.03831,  1.16425,  1.16426,  1.22635,  1.27875,  1.31906,  1.32027,  1.34822,  1.3568 ,  1.40298,  1.5652 ,  1.80587,  1.8077 ,  1.96432,  2.18166,  2.20809,  2.20838,  2.26253]),
 array([[ 6.13815e-01, -4.33638e-04, -1.33876e-17, ...,  7.71511e-15, -2.98540e-02,  2.41250e-02],
        [ 3.71379e-02, -1.05538e-01, -5.90960e-04, ...,  4.72401e-02, -8.46591e-02,  6.89496e-02],
        [ 3.41644e-02, -1.16967e-01,  3.51909e-02, ...,  1.32987e-01, -2.19688e-01,  1.82362e-01],
        ...,
        [ 1.66368e-04, -1.57966e-01,  5.90960e-04, ..., -4.72401e-02,  8.51565e-02,  6.82638e-02],
        [ 1.13370e-04, -1.58796e-01, -3.39523e-16, ..., -5.69344e-15,  2.16654e-0

In [55]:
A = LS.A.copy()
from sksparse.cholmod import cholesky
t0 = perf_counter()
factor = cholesky(A.tocsc())
#print(factor.L().nnz)
t1 = perf_counter()

x = factor.solve_A(LS.b)
t2 = perf_counter()
print(np.linalg.norm(LS.A@x-LS.b))

del factor 
print("factor", t1-t0)
print("solve", t2-t1)

9.636319129336722e-14
factor 0.21011740000039936
solve 0.00984249999964959


In [56]:
import pyMKL

A = LS.A
t0 = perf_counter()
solver = pyMKL.pardisoSolver(A, mtype=-2)
solver.factor()
t1 = perf_counter()

x = solver.solve(LS.b)
t2 = perf_counter()
print(np.linalg.norm(LS.A@x-LS.b))

solver.clear()
print("factor", t1-t0)
print("solve", t2-t1)

8.263242718678057e-14
factor 0.33144880000008925
solve 0.02722330000005968


In [59]:
from pymklpardiso import PardisoSolver
#A = scipy.sparse.csr_matrix(scipy.sparse.triu(LS.A, format='csr'))
#A.sort_indices()
A = LS.A

t0 = perf_counter()
solver = PardisoSolver(A, mtype=11)
t1 = perf_counter()

x = solver.solve(LS.b)
t2 = perf_counter()
print(np.linalg.norm(LS.A@x-LS.b))

#solver.clear()
print("factor", t1-t0)
print("solve", t2-t1)

9.263037957997925e-14
factor 0.2840214999996533
solve 0.01103140000031999


In [42]:
t = time.time()
operators.make_solver(LS.A.tocsc(), spd=True)
print(time.time()-t)

2.2262959480285645


In [32]:
import sksparse.cholmod
print(sksparse.cholmod.__file__)

/home/wolfman/miniforge3/envs/sci/lib/python3.12/site-packages/sksparse/cholmod.cpython-312-x86_64-linux-gnu.so


In [9]:
%whos

Variable       Type                          Data/Info
------------------------------------------------------
A              csr_matrix                    <Compressed Sparse Row sp<...>41672)	1.6348003848004846
Fh             ndarray                       443857: 443857 elems, type `float64`, 3550856 bytes (3.3863601684570312 Mb)
Inductor       function                      <function Inductor at 0x7f80082c2a20>
Kh             csc_matrix                    <Compressed Sparse Column<...>)	-3.1725797135722214e-08
LS             RestrictedLinearSystem        <pyiga.assemble.Restricte<...>object at 0x7f7fa9cdab10>
LinOp          function                      <function aslinearoperator at 0x7f7fe01868e0>
M              MultiPatch                    <pyiga.topology.MultiPatc<...>object at 0x7f7fa8644a10>
MB             MultiBasis                    <pyiga.assemble.MultiBasi<...>object at 0x7f80082ac750>
adaptive       module                        <module 'pyiga.adaptive' <...>pyiga/pyiga/ada

In [10]:
print("A:", A.data.nbytes/1024**3,
      A.indices.nbytes/1024**3,
      A.indptr.nbytes/1024**3)

print("Kh:", Kh.data.nbytes/1024**3,
      Kh.indices.nbytes/1024**3,
      Kh.indptr.nbytes/1024**3)

A: 0.38034821301698685 0.19017410650849342 0.0016453638672828674
Kh: 0.3822721317410469 0.19113606587052345 0.0016534999012947083
